# T2V-1K · AMD TokenFactory analogies v2

For a class the generator never saw, the TokenFactory agent names the known classes it looks like, and the classifier row is built from their learned rows. v1 (shipped) showed the agent 20 bare candidate names and asked once. v2 shows the target's description and 40 candidates **with** their descriptions, asks what looks alike in a 32×32 photo, and asks 3 times (self-consistency).

Quality is measured on 100 seen classes treated as new (their true rows are known), the blend weight is chosen there, and the official test tasks are scored once.

**Before running:** add your key as a Colab secret named `TOKENFACTORY_API_KEY` (🔑 icon on the left, enable notebook access). **Runtime → Change runtime type → T4 GPU**, then run the cell (~30-40 min; if it stops, run it again: saved TokenFactory answers are reused). It prints the tables and downloads `neighbors_imagenet_v2.json` and `t2v_analogies_v2_results.json`.

In [ ]:
#@title **T2V-1K · TokenFactory analogies v2** (Colab, T4 GPU, needs the TOKENFACTORY_API_KEY secret, ~25 min)
# For a class the generator never saw, AMD TokenFactory names the known classes it looks like, and the
# classifier row is built from their learned rows. v1 (shipped) showed the agent 20 bare candidate names
# and asked once. v2 shows the target's description and 40 candidates *with* their descriptions, asks
# what looks alike in a 32x32 photo, and asks 3 times (self-consistency) to remove the randomness.
# Quality is measured on 100 seen classes treated as new (their true rows are known), the blend weight
# is chosen there, and the official test tasks (100 per split) are scored once: shipped v1 vs v2.
import os, sys, subprocess, json, random, time, types, warnings, re
os.environ.update(TRANSFORMERS_VERBOSITY="error", HF_HUB_DISABLE_PROGRESS_BARS="1", TOKENIZERS_PARALLELISM="false")
warnings.filterwarnings("ignore")
REPO_DIR = "/content/t2v-text-to-vision"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/kalidolinux-sudo/t2v-text-to-vision", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--root-user-action=ignore", "pyarrow"], check=False)

key = None
try:
    from google.colab import userdata
    key = userdata.get("TOKENFACTORY_API_KEY")
except Exception:
    pass
if not key:
    from getpass import getpass
    key = getpass("AMD TokenFactory API key: ").strip()
os.environ["TOKENFACTORY_API_KEY"] = key
del key

from concurrent.futures import ThreadPoolExecutor
import torch, torch.nn.functional as F
import transformers; transformers.logging.set_verbosity_error()
from app import RELEASE_URL, download
from t2v import target, tokenfactory
from t2v.generate import load_generator, generate_flat
from t2v.neighbors import mix_rows, shortlist
from t2v.text_embed import embed_names
from t2v.data import _load_imagenet, MEAN, STD
from t2v.zoo import make_tasks

T0 = time.time()
def log(*a):
    print(f"[{(time.time() - T0) / 60:4.1f} min]", *a, flush=True)

DEV = "cuda" if torch.cuda.is_available() else "cpu"
CKPT = "checkpoints/t2v_1k.pt"
if not os.path.exists(CKPT):
    download(RELEASE_URL, CKPT)
gen, tok, ck = load_generator(CKPT, DEV)
names, seen = ck["names"], sorted(ck["train_classes"])
descs = ck["descriptions"]
sp = json.load(open("data/imagenet_split.json"))
meta = {**sp, "holdout": sorted(sp["holdout_random"] + sp["holdout_hard"])}
v1 = json.load(open("data/neighbors_imagenet.json"))
val_names = [n for n in v1["val_classes"] if n in names]
hold_names = [names[c] for c in meta["holdout"]]
E = embed_names(names, DEV, ensemble=True, descriptions=descs)   # the generator's condition
T = ck["row_table"].float().to(DEV)                               # zoo-mean learned rows (seen classes)
log("device", DEV, "| agent:", tokenfactory.model_name(), "| validation classes", len(val_names), "| held-out", len(hold_names))

# ---------------------------------------------------------------- 1. TokenFactory analogies v2
SYSTEM = (
    "You are a computer-vision expert helping a tiny image classifier that sees 32x32 pixel photos. For each "
    "target class you get its visual description and a list of candidate classes with descriptions. Pick the 2 to 4 "
    "candidates whose photos look MOST like photos of the target at that tiny size. What matters: overall shape and "
    "silhouette, dominant colours, texture, and the typical background or scene. Biological or functional relation "
    "matters only when it shows in the photo (a killer whale looks like a dolphin or a submarine, not like a cow). "
    "Give weights that sum to 1 (larger = more alike). Use candidate names copied exactly. "
    "Reply with JSON only: {\"<target>\": {\"<candidate>\": <weight>, ...}, ...}"
)
K_CAND, SAMPLES, BATCH = 40, 3, 5
seen_names = [names[c] for c in seen]
E_seen = E[seen]
name_idx = {n: i for i, n in enumerate(names)}


def short(n):
    d = (descs.get(n) or [""])[0]
    return " ".join(d.split()[:14])


VAL_SET = set(val_names)


def candidates_for(targets):
    # candidates are known classes; a validation class (a seen class treated as new) never sees the other
    # validation classes as candidates, so its look-alikes are classes whose rows stay known in the test below
    idx = [name_idx[t] for t in targets]
    lists = shortlist(E[idx], E_seen, seen_names, K_CAND + len(VAL_SET), exclude=targets)
    return {t: [c for c in l if c not in VAL_SET][:K_CAND] for t, l in zip(targets, lists)}


def ask_once(chunk, cands, temperature):
    lines = []
    for t in chunk:
        lines.append(f"Target: {t} ({short(t)})\nCandidates:\n" + "\n".join(f"- {c}: {short(c)}" for c in cands[t]))
    data = {}
    for attempt in range(8):  # TokenFactory rate limit (429): wait and retry with growing pauses
        try:
            text = tokenfactory.chat([{"role": "system", "content": SYSTEM}, {"role": "user", "content": "\n\n".join(lines)}],
                                     temperature=temperature, max_tokens=3000, timeout=240)
            m = re.search(r"\{.*\}", text or "", re.S)
            data = json.loads(m.group(0) if m else text)
            break
        except Exception as e:
            wait = min(120, 15 * 2 ** attempt) if "429" in str(e) else 5
            print(f"  retry in {wait}s: {str(e)[:60]}", flush=True)
            time.sleep(wait)
    out = {}
    lookup = {str(k).split(" (")[0].strip().lower(): v for k, v in data.items()}
    for t in chunk:
        picks = data.get(t) or lookup.get(t.lower()) or {}
        allowed = {c.lower(): c for c in cands[t]}
        clean = {}
        for c, w in (picks.items() if isinstance(picks, dict) else []):
            c = allowed.get(str(c).split(" (")[0].strip().lower())
            try:
                w = float(w)
            except (TypeError, ValueError):
                continue
            if c and w > 0:
                clean[c] = clean.get(c, 0) + w
        if clean:
            z = sum(clean.values()); out[t] = {c: w / z for c, w in clean.items()}
    return out


VOTES_PATH = "/content/analogy_votes_v2.json"   # raw answers, saved after every call: a rerun resumes


def analogies_v2(targets, path):
    store = json.load(open(path)) if os.path.exists(path) else {}
    todo = [t for t in targets if t not in store]
    if not todo:
        return store
    cands = candidates_for(todo)
    done = json.load(open(VOTES_PATH)) if os.path.exists(VOTES_PATH) else {}   # "sample|first target" -> answers
    jobs = [(todo[i:i + BATCH], s) for s in range(SAMPLES) for i in range(0, len(todo), BATCH)]
    jobs = [j for j in jobs if f"{j[1]}|{j[0][0]}" not in done]
    print(f"  {len(jobs)} TokenFactory calls to make ({len(done)} already saved)", flush=True)

    def run(j):
        got = ask_once(j[0], cands, 0.0 if j[1] == 0 else 0.7)
        if got:
            done[f"{j[1]}|{j[0][0]}"] = got
            json.dump(done, open(VOTES_PATH, "w"))
        return got

    with ThreadPoolExecutor(2) as ex:  # 2 at a time stays under the rate limit
        for k, _ in enumerate(ex.map(run, jobs)):
            if k % 10 == 9:
                print(f"  {k + 1}/{len(jobs)} calls", flush=True)
    votes = {t: [] for t in todo}
    for got in done.values():
        for t, p in got.items():
            if t in votes:
                votes[t].append(p)
    for t, vs in votes.items():  # self-consistency: average the samples, keep the 4 strongest look-alikes
        if not vs:
            continue
        acc = {}
        for p in vs:
            for c, w in p.items():
                acc[c] = acc.get(c, 0) + w / len(vs)
        top = dict(sorted(acc.items(), key=lambda x: -x[1])[:4])
        z = sum(top.values()); store[t] = {c: round(w / z, 4) for c, w in top.items()}
    json.dump(store, open(path, "w"), indent=1)
    return store


V2_PATH = "/content/neighbors_imagenet_v2.json"
v2 = analogies_v2(val_names + hold_names, V2_PATH)
log(f"TokenFactory analogies v2: {len(v2)}/{len(val_names) + len(hold_names)} classes, {tokenfactory.STATS['calls']} calls, {tokenfactory.STATS['seconds']:.0f} s")
for t in ["killer whale", "zebra", "grand piano", "school bus"]:
    if t in v2:
        print(f"  {t:14s} v1: {v1.get('holdout', {}).get(t) or v1.get('val', {}).get(t)}\n  {'':14s} v2: {v2[t]}")

# ---------------------------------------------------------------- 2. quality on 100 seen classes treated as new
PER, FIT, CV_LO, CV_HI = 350, 160, 300, 350
tr = _load_imagenet("train", "data", PER)
val_ids = [name_idx[n] for n in val_names]
y_all = tr["y"]; rank = torch.zeros_like(y_all); cnt = {}
for i, c in enumerate(y_all.tolist()):
    rank[i] = cnt.get(c, 0); cnt[c] = rank[i].item() + 1
keep = torch.isin(y_all, torch.tensor(val_ids)) & (rank >= CV_LO) & (rank < CV_HI)   # images the zoo never used
X, Y = tr["x"][keep], y_all[keep].to(DEV)
del tr
rng0 = random.Random(0)
flats = [generate_flat(gen, tok, rng0.sample(seen, 10), steps=5, body_noise=0.0, gate="auto")[0] for _ in range(8)]
net = target.load_flat(target.build(ck["arch"]), torch.stack(flats).mean(0).cpu()).to(DEV).eval()
with torch.no_grad():
    FX = torch.cat([net.layers(torch.relu(net.bn1(net.conv1((X[i:i + 4096].to(DEV).float() / 255 - MEAN.to(DEV)) / STD.to(DEV))))).mean((2, 3))
                    for i in range(0, len(X), 4096)])
log("validation photos", tuple(FX.shape))

others = [c for c in seen if c not in set(val_ids)]
A = E[others]
W = torch.linalg.solve(A.T @ A + 3.0 * torch.eye(A.shape[1], device=DEV), A.T @ T[others])   # ridge without the validation classes
name2id = {names[c]: c for c in others}


def rows_for(store, ids, beta):
    r = E[ids] @ W
    out = []
    for j, c in enumerate(ids):
        nb = mix_rows(store.get(names[c]), T, name2id)
        out.append(r[j] if nb is None else (1 - beta) * r[j] + beta * nb)
    return torch.stack(out)


def acc_rows(rows, cls):
    ct = torch.tensor(cls, device=DEV); m = torch.isin(Y, ct)
    lab = torch.searchsorted(ct.sort().values, Y[m]); p = ct.argsort()
    return ((FX[m] @ rows[p, :-1].T + rows[p, -1]).argmax(1) == lab).float().mean().item()


rng = random.Random("val")
tasks = [rng.sample(val_ids, 10) for _ in range(200)]
mean = lambda v: sum(v) / len(v)
BETAS = [i / 10 for i in range(11)]
quality = {}
for vname, store in [("v1 (shipped)", v1["val"]), ("v2", v2)]:
    covered = [c for c in val_ids if mix_rows(store.get(names[c]), T, name2id) is not None]
    nb_only = torch.stack([mix_rows(store.get(names[c]), T, name2id) for c in covered])
    truth = T[covered]
    cos = F.cosine_similarity(nb_only, truth, dim=-1).mean().item()
    ret = (F.normalize(nb_only, dim=-1) @ F.normalize(truth, dim=-1).T).argmax(1).eq(torch.arange(len(covered), device=DEV)).float().mean().item()
    by_beta = {b: mean([acc_rows(rows_for(store, c, b), c) for c in tasks]) for b in BETAS}
    quality[vname] = {"answered": len(covered), "row_cosine": cos, "row_retrieval": ret, "accuracy_by_beta": by_beta,
                      "analogies_only_accuracy": by_beta[1.0], "best_beta": max(by_beta, key=by_beta.get)}
print("\n=== analogy quality on 100 seen classes treated as new (200 tasks of 10, photos never used by the zoo) ===")
print(f"  ridge from text alone: accuracy {quality['v1 (shipped)']['accuracy_by_beta'][0.0]:.4f}")
for vname, q in quality.items():
    print(f"  {vname:13s} answered {q['answered']:3d}  row cosine {q['row_cosine']:.4f}  row retrieval {q['row_retrieval']:.3f}  "
          f"analogies alone {q['analogies_only_accuracy']:.4f}  best blend beta {q['best_beta']} -> {q['accuracy_by_beta'][q['best_beta']]:.4f}")
    print("     accuracy by beta: " + "  ".join(f"{b}: {a:.3f}" for b, a in q["accuracy_by_beta"].items()))
beta2 = quality["v2"]["best_beta"]

# ---------------------------------------------------------------- 3. official test tasks, scored once
val = _load_imagenet("test", "data", 260)
val_x, val_y = val["x"], val["y"]
tnet = target.build(ck["arch"]).to(DEV).eval()


def acc_flat(flat, cls):
    m = torch.isin(val_y, torch.tensor(cls))
    x = (val_x[m].to(DEV).float() / 255 - MEAN.to(DEV)) / STD.to(DEV)
    lab = torch.tensor([cls.index(int(v)) for v in val_y[m]], device=DEV)
    target.load_flat(tnet, flat)
    with torch.no_grad():
        return (tnet(x)[:, :len(cls)].argmax(1) == lab).float().mean().item()


nb_v1 = dict(gen.neighbors)
nb_v2 = {**gen.neighbors, "store": {**v2}, "llm_beta": beta2}
official, per_task = {}, {}
cols = ["eval_unseen", "eval_novel", "eval_novel_hard"]
for split in cols:
    tsk = make_tasks(types.SimpleNamespace(split=split, seed=0, n=100, unseen_frac=0.5, min_k=10, n_train_ref=1000), meta, 10)[:100]
    res = {}
    for cls in tsk:
        for label, nb, prior in [("ridge only (no analogies)", nb_v1, "ridge"), ("shipped: TokenFactory analogies v1", nb_v1, "llm"),
                                 (f"TokenFactory analogies v2 (beta {beta2})", nb_v2, "llm")]:
            gen.neighbors = nb
            torch.manual_seed(0)
            f = generate_flat(gen, tok, cls, steps=5, emb=E[cls], body_noise=0.0, gate="auto", prior=prior)[0]
            res.setdefault(label, []).append(acc_flat(f, cls))
    gen.neighbors = nb_v1
    official[split] = {k: mean(v) for k, v in res.items()}; per_task[split] = res
    log(split, "  ".join(f"{k} {v:.3f}" for k, v in official[split].items()))

print("\n" + "=" * 100)
print("T2V-1K: TokenFactory analogies v1 vs v2 | official test tasks, 100 per split, 10 classes, ImageNet validation photos")
print(f"{'':48s}" + "".join(f"{c:>12s}" for c in ["half new", "all new", "hard"]))
for k in official["eval_novel"]:
    print(f"{k:48s}" + "".join(f"{official[c][k]:12.3f}" for c in cols))
new_k = f"TokenFactory analogies v2 (beta {beta2})"
for c in cols:
    d = torch.tensor(per_task[c][new_k]) - torch.tensor(per_task[c]["shipped: TokenFactory analogies v1"])
    print(f"  {c:16s} v2 - v1: {d.mean().item():+.4f} ± {(d.std() / len(d) ** 0.5).item():.4f} (s.e.)  wins {int((d > 0).sum())}  losses {int((d < 0).sum())}")
print("=" * 100)

json.dump({"quality": quality, "beta_v2": beta2, "official": official, "per_task": per_task,
           "tokenfactory_calls": tokenfactory.STATS}, open("/content/t2v_analogies_v2_results.json", "w"), indent=1)
log("saved /content/neighbors_imagenet_v2.json and /content/t2v_analogies_v2_results.json")
try:
    from google.colab import files
    files.download("/content/t2v_analogies_v2_results.json"); files.download(V2_PATH)
except Exception:
    pass
